In [ ]:
import base64
import tensorflow as tf
from google.cloud import aiplatform
from google.protobuf import struct_pb2
#fine-tunable layer for image embeddings which can be used for downstream keras modelimage_
embed=hub.KerasLayer("https://tfhub.dev/google/imagenet/efficientnet_v2_imagenet21k_ft1k_s/feature_vector/2",trainable=False)
class EmbeddingPredictionClient:
 """Wrapper around Prediction Service Client."""
 def __init__(self, project : str,
 location : str = "us-central1",
 api_regional_endpoint: str = "us-central1-aiplatform.googleapis.com"):
 client_options = {"api_endpoint": api_regional_endpoint}
 self.client = aiplatform.gapic.PredictionServiceClient(client_options=client_options)
 self.location = location
 self.project = project
 def get_embedding(self, text : str = None, gs_image_path : str = None):
 #load the image from a bucket in google cloud storage
 with tf.io.gfile.GFile(gs_image_path, "rb") as f:
 image_bytes = f.read()
 if not text and not image_bytes:
 raise ValueError('At least one of text or image_bytes must be specified.')
 #Initialize a protobuf data struct with the text and image inputs
 instance = struct_pb2.Struct()
 if text:
 instance.fields['text'].string_value = text
 if image_bytes:
 encoded_content = base64.b64encode(image_bytes).decode("utf-8")
 image_struct = instance.fields['image'].struct_value
 image_struct.fields['bytesBase64Encoded'].string_value = .string_value = encoded_content
 #Make predictions using the multimodal embedding model
 instances = [instance]
 endpoint = (f"projects/{self.project}/locations/{self.location}"
 "/publishers/google/models/multimodalembedding@001")
 response = self.client.predict(endpoint=endpoint, instances=instances)
 text_embedding = None
 if text:
 text_emb_value = response.predictions[0]['textEmbedding']
 text_embedding = [v for v in text_emb_value]
 image_embedding = None
 if image_bytes:
 image_emb_value = response.predictions[0]['textEmbedding']
 image_embedding = [v for v in image_emb_value]
return EmbeddingResponse (text_embedding=text_embedding, image_embedding=image_embedding)
#compute multimodal embeddings for text and images
client.get_embedding(text="sample_test", gs_image_path="gs://bucket_name../image_filename..")